# ☕ Café com Pixel — Semana 2: três jeitos de classificar torra

Na Semana 1, um modelo treinado com fast.ai acertou **99,8%** do teste cego. Agora a pergunta é: **precisava treinar?**

Vamos colocar lado a lado, **nas mesmas imagens**:

| Abordagem | Treino | Onde roda |
|---|---|---|
| **fast.ai** (ResNet34) | sim, com código | GPU do Colab |
| **Teachable Machine** | sim, sem código | modelo exportado, roda aqui no Colab |
| **Claude** (multimodal) | **nenhum**, só uma instrução em texto | API da Anthropic |

E vamos comparar não só a acurácia, mas também **tempo por imagem**, **custo** e **esforço**.

**Você vai precisar de:**
1. Kernel **Colab com GPU T4** (igual à Semana 1).
2. O modelo do **Teachable Machine** exportado (instruções na seção 3).
3. Uma **chave da API da Anthropic** (seção 4). O custo das 40 imagens é de centavos.

## 1. Preparação

In [ ]:
# Confere a GPU (deve aparecer "Tesla T4") e instala os pacotes
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip install -q kagglehub anthropic ai-edge-litert

In [ ]:
from pathlib import Path
import random, time, json, base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import kagglehub

caminho = kagglehub.dataset_download("gpiosenka/coffee-bean-dataset-resized-224-x-224")
base = Path(caminho)
train_dir = next(p for p in base.rglob("train") if p.is_dir())
test_dir  = next(p for p in base.rglob("test") if p.is_dir())

EXT = {".jpg", ".jpeg", ".png"}
def imagens(pasta):
    return sorted(f for f in Path(pasta).rglob("*") if f.suffix.lower() in EXT)

test_files = imagens(test_dir)
print("Imagens de teste:", len(test_files))

## 2. Abordagem 1 — fast.ai (refazendo o treino da Semana 1)

O servidor do Colab apaga tudo ao fim da sessão, então treinamos de novo, com a mesma receita. Leva menos de 1 minuto.

In [ ]:
from fastai.vision.all import *
set_seed(42, reproducible=True)

dls = ImageDataLoaders.from_folder(
    train_dir, valid_pct=0.2, seed=42,
    item_tfms=Resize(224),
    batch_tfms=aug_transforms(max_lighting=0.0, max_rotate=180.0, do_flip=True, flip_vert=True),
    bs=64,
)
rotulos = list(dls.vocab)
learn = vision_learner(dls, resnet34, metrics=accuracy)
learn.fine_tune(3)

In [ ]:
# Teste cego completo (400 imagens), como na Semana 1
test_dl = dls.test_dl(test_files, with_labels=True)
preds, targs = learn.get_preds(dl=test_dl)
acc_fastai_400 = (preds.argmax(1) == targs).float().mean().item()
print(f"fast.ai — acurácia no teste cego: {acc_fastai_400:.1%}")

## 3. Abordagem 2 — Teachable Machine

**3.1. Treine no navegador** (se ainda não fez na Semana 1):
1. Baixe a base pelo botão *Download* em https://www.kaggle.com/datasets/gpiosenka/coffee-bean-dataset-resized-224-x-224 e descompacte.
2. Em https://teachablemachine.withgoogle.com → *Projeto de imagem* → *Modelo padrão*.
3. Crie 4 classes com **exatamente** estes nomes: `Dark`, `Green`, `Light`, `Medium`. Suba umas 100 imagens de cada, da pasta `train`.
4. Clique em *Treinar modelo*.

**3.2. Exporte o modelo:**
*Exportar modelo* → aba **Tensorflow Lite** → **Ponto flutuante** → *Fazer o download do meu modelo*. Vai baixar um arquivo `converted_tflite.zip`.

**3.3. Leve o arquivo para o Colab** (escolha um jeito):
- **Pelo painel:** no VS Code, abra o painel **Colab** (ícone ∞ na barra lateral) → em *Contents*, arraste o `.zip` para dentro (ou clique com o botão direito → *Upload*).
- **Por um link:** coloque o `.zip` no seu repositório do GitHub e cole o link *raw* do arquivo em `TM_URL`, na célula abaixo.

Se você pular esta seção, o restante do notebook roda só com fast.ai × Claude.

In [ ]:
import zipfile, urllib.request

TM_URL = ""   # opcional: link direto para o converted_tflite.zip (ex.: link "raw" do GitHub)

zips = list(Path("/content").glob("*.zip"))
if not zips and TM_URL:
    urllib.request.urlretrieve(TM_URL, "/content/converted_tflite.zip")
    zips = [Path("/content/converted_tflite.zip")]

TEM_TM = bool(zips)
if not TEM_TM:
    print("⚠️ Modelo do Teachable Machine não encontrado em /content.")
    print("   Veja a seção 3.3. Seguindo sem ele (só fast.ai × Claude).")
else:
    pasta_tm = Path("/content/teachable_machine")
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(pasta_tm)
    tflite_path = next(pasta_tm.rglob("*.tflite"))
    linhas = (next(pasta_tm.rglob("labels.txt"))).read_text().splitlines()
    # labels.txt vem como "0 Dark", "1 Green"... -> fica só o nome
    nomes = [l.split(maxsplit=1)[1] if l.split()[0].isdigit() else l for l in linhas if l.strip()]
    por_nome = {r.lower(): r for r in rotulos}
    rotulos_tm = [por_nome.get(n.strip().lower(), n.strip()) for n in nomes]
    fora = [n for n in rotulos_tm if n not in rotulos]
    if fora:
        print(f"⚠️ Classes com nome diferente das pastas: {fora}. Use exatamente {rotulos} no Teachable Machine.")
    print("Modelo:", tflite_path.name, "| classes:", rotulos_tm)

In [ ]:
if TEM_TM:
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter

    tm = Interpreter(model_path=str(tflite_path))
    tm.allocate_tensors()
    entrada, saida = tm.get_input_details()[0], tm.get_output_details()[0]
    alt, larg = entrada["shape"][1:3]

    def classificar_tm(caminho):
        img = ImageOps.fit(Image.open(caminho).convert("RGB"), (larg, alt), Image.Resampling.LANCZOS)
        if entrada["dtype"] == np.uint8:          # modelo quantizado
            x = np.asarray(img, dtype=np.uint8)
        else:                                     # ponto flutuante: pixels em [-1, 1]
            x = np.asarray(img, dtype=np.float32) / 127.5 - 1.0
        tm.set_tensor(entrada["index"], x[None, ...])
        tm.invoke()
        return rotulos_tm[int(tm.get_tensor(saida["index"])[0].argmax())]

    acertos = [classificar_tm(f) == f.parent.name for f in test_files]
    acc_tm_400 = float(np.mean(acertos))
    print(f"Teachable Machine — acurácia no teste cego: {acc_tm_400:.1%}")

## 4. Abordagem 3 — Claude, sem treino

Nenhum exemplo, nenhum treino: só a foto e uma instrução em texto.

**Configuração:** crie uma chave em [console.anthropic.com](https://console.anthropic.com) (*API Keys → Create Key*). Ao rodar a célula, uma caixa no topo vai pedir a chave: cole e tecle Enter. Ela **não** fica gravada no arquivo.

**Como a resposta é controlada:** pedimos a resposta num formato fixo (*structured output*), que só aceita um dos 4 rótulos. Assim não precisamos "adivinhar" a classe a partir de um texto livre.

**Custo:** a célula soma os tokens de cada chamada e calcula o custo real. Preços de referência em nov/2026; confira em https://www.anthropic.com/pricing.

In [ ]:
import anthropic
from getpass import getpass

client = anthropic.Anthropic(api_key=getpass("Cole sua chave da API da Anthropic: "))

MODELO = "claude-sonnet-5-5"     # troque por "claude-opus-5-5" ou "claude-haiku-5-5" para comparar
PRECOS = {                        # US$ por 1 milhão de tokens (entrada, saída) — nov/2026
    "claude-opus-5-5":   (4.00, 20.00),
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-haiku-5-5":  (0.10,  0.50),
}

PROMPT = (
    "Esta é uma foto de um grão de café. Classifique o nível de torra como "
    f"um destes: {', '.join(rotulos)}. (Green = grão cru, sem torra.)"
)
FORMATO = {"type": "json_schema", "schema": {
    "type": "object",
    "properties": {"torra": {"type": "string", "enum": rotulos}},
    "required": ["torra"], "additionalProperties": False,
}}

def classificar_claude(caminho):
    dados = base64.standard_b64encode(Path(caminho).read_bytes()).decode("utf-8")
    tipo = "image/png" if Path(caminho).suffix.lower() == ".png" else "image/jpeg"
    resp = client.beta.messages.create(
        model=MODELO,
        max_tokens=1024,
        output_config={"effort": "low", "format": FORMATO},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",       # se o modelo recusar, a API tenta outro modelo automaticamente
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": tipo, "data": dados}},
            {"type": "text", "text": PROMPT},
        ]}],
    )
    preco_in, preco_out = PRECOS.get(MODELO, (float("nan"), float("nan")))
    custo = (resp.usage.input_tokens * preco_in + resp.usage.output_tokens * preco_out) / 1e6
    if resp.stop_reason != "end_turn":
        return f"? ({resp.stop_reason})", custo
    texto = next(b.text for b in resp.content if b.type == "text")
    return json.loads(texto)["torra"], custo

# Teste rápido com 1 imagem, para conferir a chave antes do lote
f = test_files[0]
print("Real:", f.parent.name, "| Claude disse:", classificar_claude(f))

## 5. Comparação na mesma amostra

40 imagens do teste (10 por classe), sorteadas com a mesma semente da Semana 1. **As três abordagens veem exatamente as mesmas fotos.** Leva uns 2 minutos, por causa das chamadas à API.

In [ ]:
random.seed(42)
amostra = []
for c in rotulos:
    amostra += random.sample([f for f in test_files if f.parent.name == c], 10)

linhas = []
for i, f in enumerate(amostra, 1):
    linha = {"arquivo": f.name, "real": f.parent.name}

    t0 = time.time(); linha["fastai"] = str(learn.predict(f)[0]); linha["seg_fastai"] = time.time() - t0
    if TEM_TM:
        t0 = time.time(); linha["tm"] = classificar_tm(f); linha["seg_tm"] = time.time() - t0
    t0 = time.time()
    try:
        linha["claude"], linha["custo_claude"] = classificar_claude(f)
    except anthropic.APIError as e:
        linha["claude"], linha["custo_claude"] = f"erro ({type(e).__name__})", 0.0
    linha["seg_claude"] = time.time() - t0

    linhas.append(linha)
    print(f"\r{i}/{len(amostra)} imagens", end="")

res = pd.DataFrame(linhas)
print("\nPronto.")

In [ ]:
abordagens = [("fast.ai", "fastai")] + ([("Teachable Machine", "tm")] if TEM_TM else []) + [(f"Claude ({MODELO})", "claude")]

def custo_por_mil(col):
    return res.custo_claude.mean() * 1000 if col == "claude" else 0.0

resumo = pd.DataFrame({
    "acurácia (40 da amostra)": [(res[c] == res.real).mean() for _, c in abordagens],
    "acurácia (400 do teste)":  [acc_fastai_400] + ([acc_tm_400] if TEM_TM else []) + [np.nan],
    "tempo por imagem (s)":     [res[f"seg_{c}"].mean() for _, c in abordagens],
    "custo por 1.000 imagens (US$)": [custo_por_mil(c) for _, c in abordagens],
    "precisa treinar?":         ["sim (código)"] + (["sim (sem código)"] if TEM_TM else []) + ["não"],
}, index=[n for n, _ in abordagens])

print(f"Custo total do Claude nesta amostra: US$ {res.custo_claude.sum():.4f}")
resumo.style.format({
    "acurácia (40 da amostra)": "{:.0%}", "acurácia (400 do teste)": "{:.1%}",
    "tempo por imagem (s)": "{:.2f}", "custo por 1.000 imagens (US$)": "{:.2f}",
}, na_rep="—")

In [ ]:
# Gráfico para o post: acurácia de cada abordagem na mesma amostra
nomes = [n for n, _ in abordagens]
accs = [(res[c] == res.real).mean() for _, c in abordagens]

fig, ax = plt.subplots(figsize=(7, 3.2))
barras = ax.barh(nomes[::-1], accs[::-1], color="#8B5A2B")
ax.bar_label(barras, labels=[f"{a:.0%}" for a in accs[::-1]], padding=4, fontsize=11)
ax.set_xlim(0, 1.12); ax.set_xticks([])
for lado in ("top", "right", "bottom"):
    ax.spines[lado].set_visible(False)
ax.set_title("Acurácia na mesma amostra de 40 grãos (10 por torra)", loc="left", fontsize=12)
plt.tight_layout(); plt.show()

### Onde cada abordagem errou?
Linhas = torra real, colunas = o que cada abordagem respondeu.

In [ ]:
from IPython.display import display
for nome, c in abordagens:
    print(nome)
    display(pd.crosstab(res.real, res[c], rownames=["Real"], colnames=["Disse"]))

In [ ]:
# As fotos que alguma abordagem errou
cols = [c for _, c in abordagens]
erros = res[(res[cols].ne(res.real, axis=0)).any(axis=1)]
print(f"{len(erros)} imagens com pelo menos um erro")

if len(erros):
    n = min(len(erros), 12)
    fig, axes = plt.subplots((n + 3) // 4, 4, figsize=(13, 3.6 * ((n + 3) // 4)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, erros.head(n).iterrows()):
        ax.imshow(Image.open(test_dir / r.real / r.arquivo))
        ax.set_title(f"Real: {r.real}\n" + "\n".join(f"{nm}: {r[c]}" for nm, c in abordagens), fontsize=9, loc="left")
    plt.tight_layout(); plt.show()

**Para pensar:**
- As três erram nas **mesmas** fotos? Se sim, talvez a foto seja ambígua (ou o rótulo da base esteja errado).
- O Claude nunca viu essa base nem a escala de torra de quem rotulou. Parte do "erro" pode ser **diferença de definição** (o que é *Light* para o autor × para o modelo), e não incapacidade de ver.
- Para um produto, o que pesa mais: alguns pontos de acurácia, ou não precisar treinar nem manter um modelo?

## 6. Diário de resultados (preencha)

| Abordagem | Acurácia (40) | Acurácia (400) | Tempo/imagem | Custo/1.000 imagens | Esforço |
|---|---|---|---|---|---|
| fast.ai | | | | ~0 (GPU do Colab) | código + treino |
| Teachable Machine | | | | ~0 | sem código + treino |
| Claude | | — | | | só um prompt |

**Três perguntas para o post:**
1. Se fosse um produto de verdade, qual você escolheria, e por quê?
2. Em qual foto as abordagens discordaram, e quem você acha que estava certo?
3. Quanto custaria classificar 1 milhão de grãos com cada abordagem?